In [357]:
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

In [358]:
songs = joblib.load("../data/processed/songs.pkl")

audio_matrix = np.load("../data/processed/audio_matrix.npy")

tfidf_matrix = sparse.load_npz("../data/processed/tfidf_matrix.npz")

vectorizer = joblib.load("../data/processed/tfidf_vectorizer.pkl")

In [359]:
print(songs.shape)
print(audio_matrix.shape)
print(tfidf_matrix.shape)

(497138, 16)
(497138, 9)
(497138, 116389)


In [360]:
user_tracks = [
    {
        "song": "30 Hours",
        "artist": "Kanye West"
    },
    {
        "song": "Dogs",
        "artist": "Pink Floyd"
    },
    {
        "song": "Drugs You Should Try It",
        "artist": "Travis Scott"
    },
    {
        "song": "Comfortably Numb",
        "artist": "Pink Floyd"
    },
    {
        "song": "Hotel California",
        "artist": "Eagles"
    }
]

In [361]:
def normalize(feature, prefix):
    if pd.isna(feature):
        return ""

    feature = feature.split(",")

    feature = [
        f"{prefix}_{f.strip().lower().replace(' ', '_')}"
        for f in feature
    ]

    return " ".join(feature)

In [362]:
favorite_genres = [
    "rock",
    "alternative rock",
    "hip hop",
    "rap"
]

favorite_artists = [
    "Kanye West",
    "Travis Scott",
    "Pink Floyd",
    "King Crimson"
]
favorite_albums = [
    "The Life of pablo",
    "Animals",
    "days before rodeo",
    "The Wall",
    "Hotel California"
]

In [363]:
artist_query = " ".join(
    normalize(artist, "artist")
    for artist in favorite_artists
)

genre_query = " ".join(
    normalize(genre, "genre")
    for genre in favorite_genres
)

album_query = " ".join(
    normalize(album, "album")
    for album in favorite_albums
)

metadata_query = artist_query + " " + genre_query + " " + album_query

In [364]:
matched_rows = []

for track in user_tracks:
    match = songs[(songs["song"].str.lower() == track["song"].lower()) & (songs["Artist(s)"].str.contains(track["artist"], case=False, na=False))]

    if not match.empty:
        matched_rows.append(match)

matched = pd.concat(matched_rows, ignore_index=False)

matched

,song,Artist(s),Genre,emotion,Album,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Tempo,Loudness (db),metadata
224941,30 Hours,Kanye West,"synthpop,pop,rap",joy,The Life Of Pablo,61,81,82,37,29,9,4,0,95,-6.71,artist_kanye_west genre_synthpop genre_pop gen...
334516,Dogs,Pink Floyd,"rock,progressive rock,hard rock",fear,Animals,58,48,32,14,5,15,8,0,109,-11.40,artist_pink_floyd genre_rock genre_progressive...
458060,Drugs You Should Try It,Travis Scott,"rap,hip hop,trap",joy,Birds In The Trap Sing McKnight,79,59,51,59,6,12,60,5,108,-8.15,artist_travis_scott genre_rap genre_hip_hop ge...
334507,Comfortably Numb,Pink Floyd,"rock,progressive rock,hard rock",sadness,The Wall,72,37,47,17,3,8,15,31,127,-12.60,artist_pink_floyd genre_rock genre_progressive...
126248,Hotel California,Eagles,"rock,pop,classic rock",joy,Hotel California (2013 Remaster),83,51,58,61,3,6,1,0,147,-9.48,artist_eagles genre_rock genre_pop genre_class...


In [365]:
indices = matched.index.to_numpy()

indices

array([224941, 334516, 458060, 334507, 126248], dtype=int64)

In [366]:
user_audio_vectors = audio_matrix[indices]

user_audio_vectors.shape

(5, 9)

In [367]:
user_audio_profile = np.mean(user_audio_vectors, axis=0)

user_audio_profile.shape

(9,)

In [368]:
user_metadata_vector = vectorizer.transform([metadata_query])

In [369]:
audio_similarity = cosine_similarity(
    user_audio_profile.reshape(1, -1),
    audio_matrix
).flatten()

audio_similarity

array([-0.35029667, -0.58334777, -0.7182373 , ...,  0.05879335,
       -0.2983858 ,  0.17394483])

In [370]:
metadata_similarity = cosine_similarity(
    user_metadata_vector,
    tfidf_matrix
).flatten()

metadata_similarity

array([0.00604043, 0.00604043, 0.00763605, ..., 0.00979987, 0.041186  ,
       0.02625175])

In [371]:
audio_similarity = (audio_similarity - audio_similarity.min()) / (audio_similarity.max() - audio_similarity.min())

metadata_similarity = (metadata_similarity - metadata_similarity.min()) / (metadata_similarity.max() - metadata_similarity.min())

In [372]:
alpha = 0.45
beta = 0.30
gamma = 0.25

popularity_score = songs["Popularity"] / 100

final_scores = alpha * audio_similarity + beta * metadata_similarity + gamma * popularity_score

final_scores[indices] = -1
final_scores

0         0.250935
1         0.201074
2         0.137806
3         0.197137
4         0.262859
            ...   
497133    0.313703
497134    0.438110
497135    0.302058
497136    0.197367
497137    0.286829
Name: Popularity, Length: 497138, dtype: float64

In [373]:
top_indices = np.argsort(final_scores)[-50:][::-1]

recommendations = songs.iloc[top_indices].copy()

recommendations["score"] = final_scores[top_indices]
recommendations["audio_score"] = audio_similarity[top_indices]
recommendations["metadata_score"] = metadata_similarity[top_indices]
recommendations["popularity_score"] = popularity_score[top_indices]
recommendations[["song", "Album","Artist(s)", "Genre", "score" , "audio_score" , "metadata_score", "popularity_score"]]

,song,Album,Artist(s),Genre,score,audio_score,metadata_score,popularity_score
334652,Comfortably Numb Live at Nassau Coliseum 1988,The Wall,Pink Floyd,"rock,progressive rock,hard rock",0.892684,0.931206,0.978804,0.72
334513,Hey You,The Wall,Pink Floyd,"rock,progressive rock,hard rock",0.866213,0.914389,0.974126,0.65
334714,Wined And Dined,Animals,Pink Floyd,"rock,progressive rock,hard rock",0.854311,0.922883,0.963378,0.60
334517,Mother,The Wall,Pink Floyd,"rock,progressive rock,hard rock",0.851260,0.904790,0.980347,0.60
225129,Freestyle,The Life Of Pablo,Kanye West,"synthpop,pop,rap",0.850666,0.885924,0.998334,0.61
225046,4U,The Life Of Pablo,Kanye West,"synthpop,pop,rap",0.833666,0.885924,1.000000,0.54
224961,Freestyle 4,The Life Of Pablo,Kanye West,"synthpop,pop,rap",0.831651,0.885924,0.993284,0.54
224922,Wolves,The Life Of Pablo,Kanye West,"synthpop,pop,rap",0.820537,0.802305,0.998334,0.64
334670,In The Flesh?,The Wall,Pink Floyd,"rock,progressive rock,hard rock",0.803017,0.818396,0.957462,0.59
334671,In The Flesh (Ii),The Wall,Pink Floyd,"rock,progressive rock,hard rock",0.803017,0.818396,0.957462,0.59


In [374]:
def diversify(df,indices,scores,max_cnt = 3,n = 20):
    recs = []
    artists = {}

    for idx in indices:
        row = df.iloc[idx]
        artist = row["Artist(s)"]

        if artists.get(artist,0) >= max_cnt:
            continue
        artists[artist] = artists.get(artist,0) + 1

        recs.append({"song" : row["song"] , "artist" : artist , "genre" : row["Genre"], "score" : scores[idx]})

        if len(recs) == n:
            break
    return pd.DataFrame(recs)

In [375]:
indices = np.argsort(final_scores)[-1000:][::-1]
recs = diversify(songs,indices,final_scores)
recs

,song,artist,genre,score
0,Comfortably Numb Live at Nassau Coliseum 1988,Pink Floyd,"rock,progressive rock,hard rock",0.892684
1,Hey You,Pink Floyd,"rock,progressive rock,hard rock",0.866213
2,Wined And Dined,Pink Floyd,"rock,progressive rock,hard rock",0.854311
3,Freestyle,Kanye West,"synthpop,pop,rap",0.850666
4,4U,Kanye West,"synthpop,pop,rap",0.833666
5,Freestyle 4,Kanye West,"synthpop,pop,rap",0.831651
6,HIGHEST IN THE ROOM,Travis Scott,"rap,hip hop,trap",0.707897
7,Pour Up,Travis Scott,"rap,hip hop,trap",0.705016
8,Green Purple,Travis Scott,"rap,hip hop,trap",0.705016
9,PNL - Le monde ou rien English Translation,PNL,"rap,hip hop",0.676603
